# Le forme, che sono meta' del mestiere

*Anatomia di un LLM* · capitolo 3, I mattoni · script `01-01`, `scripts/01_01_tensori_e_shape.py`

Un tensore e' una griglia di numeri con una forma. Quasi tutti gli errori che
si fanno costruendo un modello sono errori di forma, e quasi tutta la fatica di
leggerne uno e' seguire come la forma cambia. Vale la pena spenderci una
pagina prima di cominciare.

Si esegue con `python3 scripts/01_01_tensori_e_shape.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_01_tensori_e_shape.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.utils import prepara, stampa_shape, titolo, verifica

prepara()

B, T, C = 2, 5, 4   # batch, tempo (token), canali (d_model)

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Come entra il testo")
x = torch.arange(B * T * C, dtype=torch.float32).reshape(B, T, C)
stampa_shape("x = (B, T, C)", x)
print(f"  elementi: {x.numel()}   memoria: {x.numel() * x.element_size()} byte")
print(f"  stride:   {x.stride()}")


Come entra il testo
-------------------
  x = (B, T, C)                (2, 5, 4)              float32
  elementi: 40   memoria: 160 byte
  stride:   (20, 4, 1)


In [4]:
titolo("Cambiare forma non sposta niente")
piatto = x.view(B * T, C)
stampa_shape("view(B*T, C)", piatto)
verifica(piatto.data_ptr() == x.data_ptr(),
         "view condivide la memoria con il tensore di partenza")
piatto[0, 0] = -1.0
verifica(x[0, 0, 0].item() == -1.0,
         "scrivere sulla view modifica anche x")
x[0, 0, 0] = 0.0


Cambiare forma non sposta niente
--------------------------------
  view(B*T, C)                 (10, 4)                float32
  [verifica superata] view condivide la memoria con il tensore di partenza
  [verifica superata] scrivere sulla view modifica anche x


In [5]:
titolo("Scambiare due assi, invece, si paga")
xt = x.transpose(1, 2)
stampa_shape("transpose(1, 2)", xt)
print(f"  contiguo: {xt.is_contiguous()}   stride: {xt.stride()}")
try:
    xt.view(B, C * T)
except RuntimeError as e:
    print(f"  view su tensore non contiguo fallisce: {type(e).__name__}")
xtc = xt.contiguous()
stampa_shape("dopo contiguous()", xtc)
verifica(xtc.data_ptr() != xt.data_ptr(),
         "contiguous() ha copiato i dati in un nuovo blocco di memoria")


Scambiare due assi, invece, si paga
-----------------------------------
  transpose(1, 2)              (2, 4, 5)              float32
  contiguo: False   stride: (20, 1, 4)
  view su tensore non contiguo fallisce: RuntimeError
  dopo contiguous()            (2, 4, 5)              float32
  [verifica superata] contiguous() ha copiato i dati in un nuovo blocco di memoria


In [6]:
titolo("Il gesto che si ripete piu' spesso")
n_head, d_head = 2, C // 2
teste = x.view(B, T, n_head, d_head).transpose(1, 2)
stampa_shape("(B, T, C) -> (B, h, T, d_h)", teste)
ricomposto = teste.transpose(1, 2).contiguous().view(B, T, C)
verifica(torch.equal(ricomposto, x),
         "split in teste e ricomposizione riportano al tensore di partenza")


Il gesto che si ripete piu' spesso
----------------------------------
  (B, T, C) -> (B, h, T, d_h)  (2, 2, 5, 2)           float32
  [verifica superata] split in teste e ricomposizione riportano al tensore di partenza


In [7]:
titolo("Sommare una cosa piccola a una grande")
pos = torch.arange(T, dtype=torch.float32).unsqueeze(1).repeat(1, C)
stampa_shape("pos = (T, C)", pos)
somma = x + pos
stampa_shape("x + pos", somma)
print("  regola: le forme si allineano da destra, (2,5,4) e (5,4) -> (2,5,4)")
verifica(torch.equal(somma[0], x[0] + pos) and torch.equal(somma[1], x[1] + pos),
         "lo stesso vettore posizionale e' stato aggiunto a entrambi gli esempi")


Sommare una cosa piccola a una grande
-------------------------------------
  pos = (T, C)                 (5, 4)                 float32
  x + pos                      (2, 5, 4)              float32
  regola: le forme si allineano da destra, (2,5,4) e (5,4) -> (2,5,4)
  [verifica superata] lo stesso vettore posizionale e' stato aggiunto a entrambi gli esempi


In [8]:
titolo("Quando non funziona")
sbagliato = torch.ones(T)
try:
    x + sbagliato
except RuntimeError:
    print(f"  x{tuple(x.shape)} + v{tuple(sbagliato.shape)}: ultimo asse 4 contro 5")
    print("  il broadcasting non allinea (T,) con l'asse dei canali: serve (T, 1)")
corretto = sbagliato.unsqueeze(1)
stampa_shape("dopo unsqueeze(1)", corretto)
stampa_shape("x + v(T,1)", x + corretto)


Quando non funziona
-------------------
  x(2, 5, 4) + v(5,): ultimo asse 4 contro 5
  il broadcasting non allinea (T,) con l'asse dei canali: serve (T, 1)
  dopo unsqueeze(1)            (5, 1)                 float32
  x + v(T,1)                   (2, 5, 4)              float32


In [9]:
titolo("Moltiplicare mille matrici insieme")
W = torch.randn(C, 3)
y = x @ W
stampa_shape("x (B,T,C) @ W (C,3)", y)
print("  matmul lascia intatti gli assi a sinistra: solo l'ultimo cambia")
verifica(torch.allclose(y[1, 2], x[1, 2] @ W, atol=1e-6),
         "il prodotto in batch coincide con il prodotto sul singolo token")


Moltiplicare mille matrici insieme
----------------------------------
  x (B,T,C) @ W (C,3)          (2, 5, 3)              float32
  matmul lascia intatti gli assi a sinistra: solo l'ultimo cambia
  [verifica superata] il prodotto in batch coincide con il prodotto sul singolo token


## Da provare

cambia B, T e C in cima e guarda come si propagano le forme.